# Exploración NASA FIRMS (VIIRS) — Galicia 2019-2024

**Autor:** Enrique Bravo — Tarea grupal: *Investigar Datasets (NASA FIRMS)*

**Fuente de datos:** portal FIRMS de NASA → menú ☰ → *Download Archived Data* → *Country Yearly Summary*.
Archivos anuales por país (sin API key): `https://firms.modaps.eosdis.nasa.gov/data/country/csv/viirs-snpp/<año>/viirs-snpp_<año>_Spain.csv`

**Sensor:** VIIRS a bordo del satélite Suomi-NPP, resolución 375 m. Es el sensor prioritario del proyecto (docs del repo).

**Contorno de Galicia:** GADM 4.1 nivel 1 (misma fuente que la rejilla de la Fase 1).

In [ ]:
import glob

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# 1. Cargar los 6 CSV anuales de España y unirlos
df = pd.concat(
    [pd.read_csv(f) for f in sorted(glob.glob("viirs-snpp_*_Spain.csv"))],
    ignore_index=True,
)
print("Detecciones España 2019-2024:", len(df))
df.head(3)

## Diccionario de columnas relevantes

| Columna | Significado |
|---|---|
| `latitude`, `longitude` | Centro del píxel de 375 m donde se detectó la anomalía térmica |
| `acq_date`, `acq_time` | Fecha y hora (UTC) de la pasada del satélite |
| `confidence` | Calidad de la detección: `l` (baja), `n` (nominal), `h` (alta) |
| `frp` | *Fire Radiative Power* (MW): intensidad del fuego |
| `daynight` | Pasada diurna (D) o nocturna (N) |
| `type` | 0 = vegetación (presunta), 2 = fuente estática (industria), 3 = agua |

In [ ]:
# 2. Recortar a Galicia con la frontera real de GADM (no un simple rectángulo,
#    para no colar focos de Asturias, León o Portugal)
esp = gpd.read_file("gadm41_ESP_1.json")
galicia = esp[esp.NAME_1 == "Galicia"].geometry.iloc[0]

g = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326"
)
g = g[g.within(galicia)].copy()
print("Detecciones dentro de Galicia:", len(g))

In [ ]:
# 3. Filtro de calidad (criterio de la documentación del proyecto):
#    - confianza nominal o alta (descarta reflejos y falsos positivos)
#    - type == 0: solo presunta vegetación (descarta chimeneas industriales
#      que el satélite ve calientes todos los días, y reflejos sobre agua)
# NOTA: usar g["type"], no g.type (choca con un atributo interno de GeoPandas)
gq = g[(g.confidence.isin(["n", "h"])) & (g["type"] == 0)].copy()
gq["acq_date"] = pd.to_datetime(gq.acq_date)
gq["year"] = gq.acq_date.dt.year
gq["month"] = gq.acq_date.dt.month
print("Tras filtro de calidad:", len(gq))
print("\nPor año:")
print(gq.year.value_counts().sort_index())

In [ ]:
# 4. Análisis exploratorio
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

gq.year.value_counts().sort_index().plot.bar(ax=axs[0, 0], color="#c0392b")
axs[0, 0].set_title("Focos por año")

meses = ["E", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
conteo_mes = gq.month.value_counts().sort_index().reindex(range(1, 13), fill_value=0)
conteo_mes.plot.bar(ax=axs[0, 1], color="#e67e22")
axs[0, 1].set_xticklabels(meses)
axs[0, 1].set_title("Estacionalidad")

esp[esp.NAME_1 == "Galicia"].boundary.plot(ax=axs[1, 0], color="black", linewidth=0.8)
gq_geo = gpd.GeoDataFrame(
    gq, geometry=gpd.points_from_xy(gq.longitude, gq.latitude), crs="EPSG:4326"
)
gq_geo.plot(ax=axs[1, 0], markersize=1.5, alpha=0.25, color="#c0392b")
axs[1, 0].set_title("Localización de focos")
axs[1, 0].set_axis_off()

gq.set_index("acq_date").resample("W").size().plot(ax=axs[1, 1], color="#8e44ad")
axs[1, 1].set_title("Focos por semana")

plt.tight_layout()
plt.show()

In [ ]:
# 5. Guardar el dataset limpio de Galicia
gq.drop(columns="geometry").to_csv("firms_galicia_2019_2024_limpio.csv", index=False)
print("Guardado firms_galicia_2019_2024_limpio.csv con", len(gq), "filas")

## Conclusiones

1. **5.132 detecciones de calidad** en Galicia 2019-2024 (de 94.364 de España).
2. **El filtrado se valida solo:** 2022 concentra el 55% de los focos y los días pico (18-19 julio 2022) coinciden con la ola de O Courel y Valdeorras. El pico de septiembre 2020 coincide con los incendios de Ourense de ese año.
3. **Estacionalidad doble:** máximo julio-septiembre (82%) + pico secundario en marzo (quemas de primavera, patrón típico gallego que el modelo deberá aprender).
4. **Concentración espacial en Ourense oriental** (O Courel, Valdeorras, Baixa Limia): coherente con la realidad, no hay artefactos urbanos.
5. Dos tercios de las detecciones son **nocturnas**: el satélite discrimina mejor de noche.
6. FRP mediana ~6 MW, máximo 475 MW: la distribución de intensidad es muy asimétrica (pocos fuegos concentran casi toda la energía).